# Coverage of the validation set

Workflow: operating conditions of the test profile, evaluation against the pool P, reference design and augmentation, numbers and figures of the manuscript. Results go to `results/`.

## 1. Setup and imports

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').exists())
sys.path.insert(0, str(root))

from src.io import coverage as cov
from src.io.runtime_paths import resolve_results_dir

results_dir = resolve_results_dir()
figure_dir = results_dir / 'figures'
figure_dir.mkdir(parents=True, exist_ok=True)

# The pool convergence check takes longest.
RUN_CONVERGENCE = True

## 2. Operating conditions of the test profile

Each condition was run twice. The coupling constraints are checked with a tolerance, because the profile holds rounded setpoints.

In [ ]:
steps = cov.load_profile_steps()
design, measured, table = cov.operating_conditions(steps)
table.to_csv(results_dir / 'D0_operating_conditions.csv', index=False)
print(f'Steps {len(steps)}, operating conditions {len(design)}, rule tolerance {cov.RULE_TOLERANCE}')
display(design.describe().T.round(3))
display(table.head())

## 3. Evaluation against the pool

In [ ]:
study = cov.run_study(design, measured)
summary = cov.summarize(study)
(results_dir / 'coverage_numbers.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
pd.Series(summary).to_frame('Value')

## 4. Sobol' reference

In [ ]:
cov.plot_sobol(study, figure_dir / 'sobol_dist.png')
display(Image(figure_dir / 'sobol_dist.png', width=700))

## 5. Largest gaps

In [ ]:
cov.plot_gap(study, figure_dir / 'cov_gap.png')
gaps = cov.largest_gaps(study)
gaps.to_csv(results_dir / 'coverage_largest_gaps.csv')
display(gaps.round(3))
display(Image(figure_dir / 'cov_gap.png', width=700))

## 6. Augmentation

In [ ]:
cov.plot_augmentation(study, figure_dir / 'cov_augment.png')
display(Image(figure_dir / 'cov_augment.png', width=700))

## 7. Pool convergence

In [ ]:
if RUN_CONVERGENCE:
    convergence = cov.pool_convergence(design)
    convergence.to_csv(results_dir / 'pool_convergence.csv', index=False)
else:
    convergence = pd.read_csv(results_dir / 'pool_convergence.csv')
convergence